# LELA60331 Week 3 Seminar · Part 2: Vector-based models of word meaning

This sheet introduces a new idea: representing the meaning of a word as a **vector** (a list of numbers), built by counting the words that occur near it.

Each **Activity** is short: replace every `???` with your own code and run the cell. Parts marked ⭐ **Challenge** are optional. Run the cells in order: later cells use variables made by earlier ones.

## Setting up

Later in this sheet we build word vectors from *Crime and Punishment*. Run this cell first: it downloads the book from Project Gutenberg (the `!` runs a command outside Python). It is the same download as in Part 1.

In [ ]:
!wget -nc https://www.gutenberg.org/files/2554/2554-0.txt

## Vectors in Python

This sheet looks at vector-based models of word meaning. A **vector** is just a list of numbers, so in Python we can store one as an ordinary list. Indexing and slicing work as for any list:

In [ ]:
a = [9, 2, 3, 5]
print(a[1])
print(a[1:3])

For vectors we can perform the operations that we learned about in our lecture. They work **element by element**: the first element of `a` with the first element of `b`, the second with the second, and so on. We use an index-based `for` loop, as in the Week 3 Python workbook.

Careful: `a + b` on two lists does **not** add them as vectors. It joins them into one longer list (try it!).

In [ ]:
# Vector addition
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]
c = []
for i in range(len(a)):
    c.append(a[i] + b[i])
print(a)
print(b)
print(c)

In [ ]:
# Vector subtraction
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]
c = []
for i in range(len(a)):
    c.append(a[i] - b[i])
print(a)
print(b)
print(c)

In [ ]:
# Dot product: multiply the vectors element by element, then add up the results
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]
c = []
for i in range(len(a)):
    c.append(a[i] * b[i])
dp = sum(c)
print(a)
print(b)
print(c)
print(dp)

**Activity 1 (cosine)**

The cosine of the angle between two vectors is their dot product divided by the product of their lengths:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|} \qquad \text{where } \|a\| = \sqrt{a \cdot a}$$

The loop adds up three dot products at once: `a` with `b`, `a` with itself, and `b` with itself. Fill in the blanks. For a square root, raise a number to the power 0.5: `x ** 0.5`.

⭐ **Challenge (optional):** change `b` so that it is the same as `a`, and run the cell again. A vector always has cosine 1 with itself.

In [ ]:
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]

dot = 0          # a . b
a_squares = 0    # a . a
b_squares = 0    # b . b
for i in range(len(a)):
    dot += ???                  # the element of a at position i, times the element of b at position i
    a_squares += ???
    b_squares += ???

length_a = ???                  # the square root of a . a
length_b = ???
cos_ab = ???                    # the dot product divided by the product of the two lengths
print(cos_ab)

### Building Word Vectors

**The big idea:** words with similar meanings tend to turn up in similar company. *walk* and *run* both appear near words like *to*, *the*, *down*, *street*. *shine* appears near *sun*, *light*, *bright*. So we can describe each word by **counting the words that occur near it**. These counts, written as a list of numbers, are the word's **vector**. Then we can use the cosine from Activity 1 to measure how similar two words are.

We will build this in small steps:

| Step | What we do |
|---|---|
| 1 | Load the whole novel as a list of words |
| 2 | Count the words and choose the 1000 most frequent as our *context words* |
| 3 | Look at a *context window* on a tiny example sentence |
| 4 | Store the counts in a *dictionary of dictionaries* |
| 5 | Put steps 3 and 4 together on the tiny example |
| 6 | Do the same for the whole novel |
| 7 | Turn the counts into vectors |
| 8 | Compare words with cosine |

Run every cell in order: each step uses variables made by the steps before it.

#### Step 1 · The novel as a list of words

This is the same cleaning as in Part 1 of the seminar (keep only the letters a–z and spaces), but this time we keep the **whole novel**, not just chapter one. Just run the cell.

In [ ]:
# Read in the book we downloaded in the Setting up cell
with open('2554-0.txt') as fp:
    c_and_p = fp.read()

# keep the novel itself: from the first chapter heading up to Project Gutenberg's end-of-book notice
start = c_and_p.find('CHAPTER I\n')
end = c_and_p.find('*** END OF THE PROJECT GUTENBERG')
text = c_and_p[start:end].lower().replace('\n', ' ')

# keep only the letters a-z and spaces , then split into a list of words
allowed = 'abcdefghijklmnopqrstuvwxyz '
kept = []
for ch in text:
    if ch in allowed:
        kept.append(ch)
words = ''.join(kept).split()

**Activity 2 (getting to know the data)**

`words` is a list of every word in the novel, in order. Fill in the blanks.

1. How many word **tokens** are there?
2. How many **different** words (types) are there?

Chapter one alone had only a few thousand tokens: the whole novel gives us much more data.

In [ ]:
print(words[:20])                              # the first 20 words

print('Number of tokens:', ???)                # hint: len()
print('Number of types:', ???)                 # hint: len() of a set()

#### Step 2 · Choosing the context words

We are going to describe each word by the words around it. Those surrounding words are called **context words**.

We won't use *every* word as a context word. Rare words (seen once or twice) tell us very little, and there are thousands of them. So we use only the **1000 most frequent** words.

First, count every word. This is the same counting pattern you have used for unigrams.

**Activity 3 (counting words)**

Fill in the blank with the counting pattern: get the current count (0 if the word is new) and add 1.

In [ ]:
word_counts = {}
for word in words:
    word_counts[word] = ???          # hint: word_counts.get(word, 0) ...

print('the:', word_counts['the'])
print('walk:', word_counts['walk'])

To find the most frequent words, we make a list of `(count, word)` pairs and sort it from largest to smallest (as in the ⭐ Challenge in the Week 2 seminar). Putting the count **first** in each pair means the sort goes by count. Just run the cell.

In [ ]:
pairs = []
for word, count in word_counts.items():
    pairs.append((count, word))
pairs = sorted(pairs, reverse=True)     # reverse=True: largest first

print(pairs[:10])                       # the 10 most frequent words

**Activity 4 (keeping the top 1000)**

Now keep just the **words** (not the counts) from the first 1000 pairs.

1. Fill in the slice so the loop only visits the first 1000 pairs.
2. Fill in what to append: we want the word, not the count.

We also put the words in a **set**, `context_set`. We will need to ask "is this a context word?" millions of times, and checking `in` a set is much faster than checking `in` a list.

Look at the output: what kind of words are at the top of the list? What kind are near the bottom?

In [ ]:
context_list = []
for count, word in pairs[???]:          # the first 1000 pairs
    context_list.append(???)            # just the word

context_set = set(context_list)

print(len(context_list), 'context words')
print('Top 20:', context_list[:20])
print('Last 20:', context_list[-20:])

#### Step 3 · The context window

Which words count as being "near" a word? We use a **window**: the words up to `window_size` positions before it and after it. With `window_size = 2`, the window around *walked* in this sentence is:

```
the   [old   man   WALKED   slowly   to]   the   house
 0      1     2      3        4      5     6      7
```

That is **two** words either side of position 3: positions 1, 2, 4 and 5 (*old, man, slowly, to*).

We'll practise on this tiny sentence before using the whole novel. Run the cell. It loops over the positions `j` from `i - 2` to `i + 2`.

In [ ]:
toy = ['the', 'old', 'man', 'walked', 'slowly', 'to', 'the', 'house']
window_size = 2

i = 3                                  # the position of our target word
print('Target word:', toy[i])
for j in range(i - window_size, i + window_size + 1):
    print(j, toy[j])

**Activity 5 (fixing the window)**

The output above has a problem: position 3 (*walked*) appears in its own window. A word is not its own context.

There are two more problems at the edges of the sentence. Try changing `i` to `0` in the cell above and running it again. Python treats `toy[-1]` as "the last item", so words from the **end** of the sentence wrongly appear. Now try `i = 7`: you get an `IndexError`, because there is no position 8 or 9.

Fill in the blanks so that a position `j` is only used if:
* it is not before the start (`j >= 0`)
* it is not past the end (`j < ` the length of the list)
* it is not the target word itself (`j != ` ...)

Check that it works for `i = 0`, `i = 3` and `i = 7`.

In [ ]:
i = 0
print('Target word:', toy[i])
for j in range(i - window_size, i + window_size + 1):
    if j >= 0 and j < ??? and j != ???:
        print(j, toy[j])

#### Step 4 · A dictionary of dictionaries

For each word we need a set of counts: how often *the* is near it, how often *old* is near it, and so on. That is a counting dictionary, just like `word_counts`. And we need one of these **for every word**.

So we keep them all in one big dictionary, `M`:
* `M['man']` is the counting dictionary for *man*
* `M['man']['old']` is how often *old* occurs near *man*

Run the cell below to see how this works, one step at a time.

In [ ]:
M_toy = {}
M_toy['man'] = {}                    # an empty counting dictionary for 'man'
print(M_toy)

# we see 'old' near 'man': add one to its count (the usual counting pattern)
M_toy['man']['old'] = M_toy['man'].get('old', 0) + 1
print(M_toy)

# we see 'the' near 'man'
M_toy['man']['the'] = M_toy['man'].get('the', 0) + 1
print(M_toy)

# we see 'old' near 'man' again
M_toy['man']['old'] = M_toy['man'].get('old', 0) + 1
print(M_toy)

print("M_toy['man']['old'] is", M_toy['man']['old'])

**Activity 6 (reading a dictionary of dictionaries)**

Using `M_toy` from the cell above, fill in the blanks to print:
1. the whole counting dictionary for *man*
2. how often *the* was seen near *man*

In [ ]:
print(???)          # 1. the counting dictionary for 'man'
print(???)          # 2. the count for 'the' near 'man'

#### Step 5 · Putting it together on the tiny sentence

Now we combine steps 3 and 4. For **every** position `i` in the sentence:
1. make an empty counting dictionary for the word at `i`, if it doesn't have one yet
2. visit each position `j` in its window (with the checks from Activity 5)
3. add one to the count for the word at `j`

**Activity 7 (counting contexts in the tiny sentence)**

Fill in the one blank with the counting pattern from Step 4. Then look at the result:
* What are the contexts of *walked*? Do they match the window in Step 3?
* Why does *the* have more contexts than the other words?

In [ ]:
M_toy = {}
window_size = 2

for i in range(len(toy)):
    word = toy[i]
    if word not in M_toy:
        M_toy[word] = {}                 # first time we see this word
    for j in range(i - window_size, i + window_size + 1):
        if j >= 0 and j < len(toy) and j != i:
            context = toy[j]
            M_toy[word][context] = ???   # add one to the count for this context

M_toy

#### Step 6 · The whole novel

This is exactly the code from Activity 7, with two changes:
* it runs over `words` (the novel) instead of `toy`
* it only counts a context if it is one of our 1000 context words from Step 2

**Activity 8 (counting contexts in the novel)**

Fill in the blank: the condition should be true only if `context` is in `context_set`. The cell may take several seconds to run, because the novel has over 200,000 words.

In [ ]:
M = {}
window_size = 2

for i in range(len(words)):
    word = words[i]
    if word not in M:
        M[word] = {}
    for j in range(i - window_size, i + window_size + 1):
        if j >= 0 and j < len(words) and j != i:
            context = words[j]
            if ???:                                     # only if context is one of our 1000 context words
                M[word][context] = M[word].get(context, 0) + 1

Let's look at the contexts of *walk*. The full dictionary is long, so the second cell sorts it (the same `(count, word)` trick as in Step 2) and shows the 15 most common contexts. Does anything surprise you?

In [ ]:
M['walk']

In [ ]:
walk_pairs = []
for context, count in M['walk'].items():
    walk_pairs.append((count, context))
sorted(walk_pairs, reverse=True)[:15]

⭐ **Challenge (optional):** change `'walk'` to another word, e.g. `'sun'` or `'mother'`, and run the cell again. Do its most common contexts tell you anything about its meaning? Why are words like *the* and *to* near the top for almost every word?

#### Step 7 · From counts to vectors

To use cosine, each word needs a **list of numbers**, and all the lists must be in the **same order** so that position 0 means the same context for every word, position 1 the same, and so on.

So we go through `context_list` in order and look up each context's count. If a context never occurs near the word, it isn't in the dictionary, so we use `.get(context, 0)` to get 0.

Here it is on the tiny sentence, with just four context words. Run it, and check the numbers against `M_toy` in Step 5.

In [ ]:
small_contexts = ['the', 'old', 'slowly', 'house']

man_vec = []
for context in small_contexts:
    man_vec.append(M_toy['man'].get(context, 0))

walked_vec = []
for context in small_contexts:
    walked_vec.append(M_toy['walked'].get(context, 0))

print('contexts:', small_contexts)
print('man:     ', man_vec)
print('walked:  ', walked_vec)

**Activity 9 (vectors for *walk* and *run*)**

Now do the same for the novel, using all 1000 words in `context_list`. The vector for *walk* is done for you. Fill in the blank to build the vector for *run*.

Both vectors should have 1000 numbers. Most of the numbers are 0. Why?

In [ ]:
walk_vec = []
for context in context_list:
    walk_vec.append(M['walk'].get(context, 0))

run_vec = []
for context in context_list:
    run_vec.append(???)                  # the count for this context near 'run'

print(len(walk_vec), len(run_vec))
print('walk:', walk_vec[:20])
print('run: ', run_vec[:20])

#### Step 8 · Comparing words with cosine

Now we have two vectors, we can compare them with exactly the cosine calculation from Activity 1. Only the names have changed: `a` is now `walk_vec` and `b` is now `run_vec`.

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|} \qquad \text{where } \|a\| = \sqrt{a \cdot a}$$

**Activity 10 (cosine of *walk* and *run*)**

Fill in the blanks. Look back at your answer to Activity 1 if you need to. Remember that the square root of `x` is `x ** 0.5`.

In [ ]:
dot = 0              # walk . run
walk_squares = 0     # walk . walk
run_squares = 0      # run . run
for i in range(len(context_list)):
    dot += ???
    walk_squares += ???
    run_squares += ???

cos_walk_run = ???
print('walk ~ run:', round(cos_walk_run, 3))

**Activity 11 (comparing *walk* with several words)**

The cell below puts Steps 7 and 8 inside a loop, so we can compare *walk* with every word in `others`. Nothing here is new: the inside of the loop is the code from Activities 9 and 10, with `run` replaced by `other`. Just fill in the one blank, the same as in Activity 9.

1. Run it. Is *walk* more similar to *run* or to *shine*? What does that tell us?
2. Add more words to `others` (e.g. `'go'`, `'went'`, `'house'`, `'money'`, `'talk'`). Which come out as most similar to *walk*? Are the results what you expected?

If a word never occurs in the novel, the `if` check skips it instead of giving a `KeyError`.

In [ ]:
others = ['run', 'shine']

for other in others:
    if other not in M:
        print(other, 'does not occur in the novel')
        continue

    # Step 7: the vector for other
    other_vec = []
    for context in context_list:
        other_vec.append(???)            # same as in Activity 9, but for other

    # Step 8: cosine of walk and other
    dot = 0
    walk_squares = 0
    other_squares = 0
    for i in range(len(context_list)):
        dot += walk_vec[i] * other_vec[i]
        walk_squares += walk_vec[i] * walk_vec[i]
        other_squares += other_vec[i] * other_vec[i]
    cos = dot / (walk_squares ** 0.5 * other_squares ** 0.5)
    print('walk ~', other, ':', round(cos, 3))

⭐ **Challenge (optional):**
* Go back to Step 6 and change `window_size` to `1`, then to `5`. Rerun Step 6 and everything after it. How do the similarities change? Why might a small window and a large window capture different kinds of similarity?
* The very frequent context words (*the*, *and*, *to*) occur near almost everything, so they make every pair of words look similar. Try leaving out the 50 most frequent words by using `pairs[50:1050]` in Activity 4. Do the results change?